# Spectrograms for TTS

This notebook covers spectrograms - the primary feature representation in TTS:

1. Fourier Transform basics
2. Short-Time Fourier Transform (STFT)
3. Linear spectrograms
4. Mel spectrograms (the key TTS feature!)
5. Converting between representations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Fourier Transform

The Fourier Transform converts a signal from the time domain to the frequency domain.

**Time domain**: Amplitude vs. time
**Frequency domain**: Amplitude vs. frequency

In [ ]:
# Create a signal with multiple frequencies
sample_rate = 22050
duration = 1.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Sum of 3 frequencies
f1, f2, f3 = 200, 500, 1000  # Hz
signal = (0.5 * np.sin(2 * np.pi * f1 * t) +
          0.3 * np.sin(2 * np.pi * f2 * t) +
          0.2 * np.sin(2 * np.pi * f3 * t))

# Compute FFT
fft = np.fft.rfft(signal)
frequencies = np.fft.rfftfreq(len(signal), 1/sample_rate)
magnitude = np.abs(fft)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Time domain
axes[0].plot(t[:500], signal[:500])
axes[0].set_title('Time Domain (Waveform)')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')

# Frequency domain
axes[1].plot(frequencies[:5000], magnitude[:5000])
axes[1].axvline(x=f1, color='r', linestyle='--', label=f'{f1} Hz')
axes[1].axvline(x=f2, color='g', linestyle='--', label=f'{f2} Hz')
axes[1].axvline(x=f3, color='b', linestyle='--', label=f'{f3} Hz')
axes[1].set_title('Frequency Domain (Spectrum)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Short-Time Fourier Transform (STFT)

Regular FFT gives us frequencies but loses time information.
STFT computes FFT on overlapping windows to get both time and frequency.

Key parameters:
- **n_fft**: Window size for FFT
- **hop_length**: Step between windows
- **win_length**: Window length (can differ from n_fft)

In [ ]:
def compute_stft(audio, n_fft=2048, hop_length=256, win_length=1024):
    """
    Compute Short-Time Fourier Transform.
    """
    # Pad the signal
    audio_padded = np.pad(audio, (n_fft // 2, n_fft // 2), mode='reflect')
    
    # Calculate number of frames
    num_frames = 1 + (len(audio_padded) - n_fft) // hop_length
    
    # Create window
    window = np.hanning(win_length)
    
    # Initialize STFT matrix
    stft = np.zeros((n_fft // 2 + 1, num_frames), dtype=np.complex64)
    
    # Compute STFT
    for i in range(num_frames):
        start = i * hop_length
        frame = audio_padded[start:start + win_length]
        
        # Apply window and zero-pad if needed
        windowed = np.zeros(n_fft)
        windowed[:len(frame)] = frame * window
        
        # FFT
        stft[:, i] = np.fft.rfft(windowed)
    
    return stft

# Create a chirp signal (frequency changes over time)
t = np.linspace(0, 2, 2 * 22050, endpoint=False)
chirp = np.sin(2 * np.pi * (100 + 500 * t) * t)

# Compute STFT
stft = compute_stft(chirp)
magnitude = np.abs(stft)

# Plot spectrogram
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(t, chirp)
axes[0].set_title('Waveform (Chirp signal)')
axes[0].set_xlabel('Time (seconds)')

# Spectrogram
img = axes[1].imshow(
    20 * np.log10(magnitude + 1e-10),
    aspect='auto',
    origin='lower',
    extent=[0, 2, 0, 22050/2],
    cmap='viridis'
)
axes[1].set_title('Spectrogram (STFT magnitude in dB)')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_ylim(0, 3000)
plt.colorbar(img, ax=axes[1], label='dB')

plt.tight_layout()
plt.show()

display(Audio(chirp, rate=22050))

## 3. Mel Spectrogram

The mel scale approximates human perception of pitch.
Humans perceive pitch differences logarithmically.

**Mel spectrograms** are the standard input/output representation for TTS!

In [ ]:
def hz_to_mel(hz):
    """Convert Hz to mel scale."""
    return 2595 * np.log10(1 + hz / 700)

def mel_to_hz(mel):
    """Convert mel scale to Hz."""
    return 700 * (10 ** (mel / 2595) - 1)

# Show Hz to Mel relationship
hz = np.linspace(0, 8000, 1000)
mel = hz_to_mel(hz)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hz, mel)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Mel')
ax.set_title('Hz to Mel Scale Conversion')
ax.grid(True)
plt.show()

print(f"100 Hz -> {hz_to_mel(100):.1f} mel")
print(f"1000 Hz -> {hz_to_mel(1000):.1f} mel")
print(f"8000 Hz -> {hz_to_mel(8000):.1f} mel")

In [ ]:
def create_mel_filterbank(n_fft, n_mels, sample_rate, f_min=0, f_max=None):
    """Create mel filterbank matrix."""
    if f_max is None:
        f_max = sample_rate / 2
    
    # Mel points
    mel_min = hz_to_mel(f_min)
    mel_max = hz_to_mel(f_max)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    
    # Convert to FFT bins
    bins = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)
    
    # Create filterbank
    n_freq = n_fft // 2 + 1
    filterbank = np.zeros((n_mels, n_freq))
    
    for i in range(n_mels):
        left, center, right = bins[i], bins[i+1], bins[i+2]
        
        # Rising edge
        for j in range(left, center):
            if center != left:
                filterbank[i, j] = (j - left) / (center - left)
        
        # Falling edge
        for j in range(center, right):
            if right != center:
                filterbank[i, j] = (right - j) / (right - center)
    
    return filterbank

# Create and visualize mel filterbank
n_mels = 80
mel_fb = create_mel_filterbank(2048, n_mels, 22050, f_max=8000)

fig, ax = plt.subplots(figsize=(12, 5))
freqs = np.linspace(0, 11025, mel_fb.shape[1])
for i in range(0, n_mels, 5):  # Plot every 5th filter
    ax.plot(freqs, mel_fb[i])
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Filter Weight')
ax.set_title(f'Mel Filterbank ({n_mels} filters)')
ax.set_xlim(0, 8000)
plt.show()

In [ ]:
def compute_mel_spectrogram(audio, sample_rate=22050, n_fft=2048, 
                            hop_length=256, n_mels=80, f_max=8000):
    """
    Compute mel spectrogram from audio.
    """
    # Compute STFT
    stft = compute_stft(audio, n_fft, hop_length)
    magnitude = np.abs(stft)
    
    # Create mel filterbank
    mel_fb = create_mel_filterbank(n_fft, n_mels, sample_rate, f_max=f_max)
    
    # Apply mel filterbank
    mel_spec = np.dot(mel_fb, magnitude)
    
    # Convert to log scale
    mel_spec = np.log(np.maximum(mel_spec, 1e-10))
    
    return mel_spec

# Compute mel spectrogram for the chirp
mel_spec = compute_mel_spectrogram(chirp)

# Compare linear and mel spectrograms
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Linear spectrogram
stft = compute_stft(chirp)
magnitude = np.abs(stft)
img1 = axes[0].imshow(
    20 * np.log10(magnitude + 1e-10),
    aspect='auto',
    origin='lower',
    cmap='viridis'
)
axes[0].set_title('Linear Spectrogram')
axes[0].set_ylabel('Frequency Bin')
plt.colorbar(img1, ax=axes[0], label='dB')

# Mel spectrogram
img2 = axes[1].imshow(
    mel_spec,
    aspect='auto',
    origin='lower',
    cmap='viridis'
)
axes[1].set_title('Mel Spectrogram (80 mel bins)')
axes[1].set_xlabel('Time Frame')
axes[1].set_ylabel('Mel Bin')
plt.colorbar(img2, ax=axes[1], label='Log magnitude')

plt.tight_layout()
plt.show()

print(f"Mel spectrogram shape: {mel_spec.shape}")
print(f"  - {mel_spec.shape[0]} mel bins")
print(f"  - {mel_spec.shape[1]} time frames")

## Summary

**Key points for TTS:**

1. **STFT** gives us time-frequency representation
2. **Mel scale** approximates human hearing
3. **Mel spectrograms** are the standard TTS feature:
   - Input to vocoders (HiFi-GAN)
   - Output of acoustic models (Tacotron2)
   - 80 mel bins is standard

**Typical TTS parameters:**
- Sample rate: 22050 Hz
- n_fft: 2048
- hop_length: 256
- n_mels: 80
- f_max: 8000 Hz